# Suite IDN — Identity: ownership and paths

A reference object's identity is its path: the route from a singleton to it through the relations that own it
(docs/OBJECT_IDENTITY.md). An adjacency marked `owner()` points back to the object's owner, through a relation whose
clauses make the route unambiguous; a key property may carry a naming pattern; a path is a nested value of
`Paths.OfRoot`, read off an object's ownership entries on demand. Step 4a of the store overhaul: the vocabulary, before
objects are identified by their paths.

In [ ]:
import { Builders as B, JSON as Json, Paths, Plain, Schemas as S } from "@mbse/schemas/Framework";
import { AttributeError, DecodeError, KeyError, LookupError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, map, owned, raises, same } from "./support.js";

/** The directory of definitions, each owning its clauses, built in `store`: a singleton wts.Features owns each
 * wts.Definition through wts.Defines, by a name its pattern gives, and a definition each wts.Clause through
 * wts.HasClause. */
function wts(store: any): any[] {
  const Text = store.Schemas.OfStr.as_schema();
  const Directory = store.Schemas.OfObject((o: any) => o.name("wts.Directory").ref(true).singleton("wts.Features"));
  const Definition = store.Schemas.OfObject((o: any) => o.name("wts.Definition").ref(true).properties([
    (t: any) => t.name("title").schema([Text]),
  ]));
  const Clause = store.Schemas.OfObject((o: any) => o.name("wts.Clause").ref(true));
  const Defines = store.Schemas.OfRelation((r: any) => (
    r
      .name("wts.Defines")
      .links([
        (t: any) => t.name("directory"),
        (t: any) => t.name("defn"),
      ])
      .properties([(t: any) => t.name("name").schema([Text]).pattern(String.raw`DEF-(\d+\d\d)`)])
      .uniques([["directory", "name"], ["defn"]])
  ));
  const HasClause = store.Schemas.OfRelation((r: any) => (
    r
      .name("wts.HasClause")
      .links([
        (t: any) => t.name("defn"),
        (t: any) => t.name("clause"),
      ])
      .properties([(t: any) => t.name("name").schema([Text])])
      .uniques([["defn", "name"], ["clause"]])
  ));
  store.Schemas.OfObject(Directory, (o: any) => o.adjacencies([(t: any) => t.name("dir").relation([Defines]).me("directory")]));
  store.Schemas.OfObject(Definition, (o: any) => o.adjacencies([
    (t: any) => t.name("def").relation([Defines]).me("defn").owner(true),
    (t: any) => t.name("clauses").relation([HasClause]).me("defn"),
  ]));
  store.Schemas.OfObject(Clause, (o: any) => o.adjacencies([(t: any) => t.relation([HasClause]).me("clause").owner(true)]));
  return [Directory, Definition, Clause, Defines, HasClause];
}

const store: any = new B.Store();
const Text = store.Schemas.OfStr.as_schema(); // the store's own
const [Directory, Definition, Clause, Defines, HasClause] = wts(store);

## IDN-01 · An adjacency marked `owner()`: its relation has two links, and clauses by which the owned object determines its owner and the owner and keys determine the owned object

In [ ]:
{
  assert(Definition.adjacencies.get("def").owner === true && Directory.adjacencies.get("dir").owner === null);
  assert([Directory, Definition, Clause, Defines, HasClause].every((s) => s.validate().length === 0));
  /** An object schema owned through a relation of `links` with `uniques`: its validation. */
  const ownerOf = (uniques: string[][], links = ["up", "down"]): string[] => {
    const relation = store.Schemas.OfRelation((r: any) => r.links(links.map((n) => (t: any) => t.name(n))).uniques(uniques));
    return store.Schemas.OfObject((o: any) => o.ref(true).adjacencies([
      (t: any) => t.name("up").relation([relation]).me("down").owner(true),
    ])).validate();
  };
  assert(ownerOf([["up"], ["down"]]).length === 0); // keyless: the owner alone determines the owned object
  assert(same(ownerOf([["down"]]), ["adjacency 'up' is its object's owner: no unique clause has 'up' without 'down', so that "
    + "the owned object determines its owner"]));
  assert(same(ownerOf([["up"]]), ["adjacency 'up' is its object's owner: no unique clause has 'down' without 'up', so that "
    + "the owner and the keys determine the owned object"]));
  assert(same(ownerOf([["up"], ["down"]], ["up", "down", "side"]), [
    "adjacency 'up' is its object's owner: its relation has two links, the owner's and the owned's"]));
  raises(TypeError, () => store.Schemas.OfObject((o: any) => o.adjacencies([(t: any) => t.relation([Defines]).me("defn").owner("yes")])),
    "property 'owner' holds bool, got str");
}

## IDN-02 · A key property's naming pattern: the values it gives and reads, and what validation refuses

In [ ]:
{
  const name = Defines.entry.properties.get("name");
  assert(name.pattern === String.raw`DEF-(\d+\d\d)` && name.name_for(7) === "DEF-007" && name.name_for(1234) === "DEF-1234");
  assert(same([name.number_of("DEF-693"), name.number_of("DEF-69"), name.number_of("FEA-693")], [693, null, null]));
  assert(new S.OfProperty.Data({ name: "k", schema: Text }).number_of("DEF-693") === null); // no pattern
  const bad = store.Schemas.OfRelation((r: any) => (
    r
      .links([(t: any) => t.name("a"), (t: any) => t.name("b")])
      .properties([
        (t: any) => t.name("k").schema([Text]).pattern("DEF-.*"),
        (t: any) => t.name("n").schema([store.Schemas.OfInt]).pattern(String.raw`(\d+)`),
      ])
  ));
  assert(same(bad.validate(), [
    "property 'k': naming pattern 'DEF-.*' is not literal text around one group of digits, e.g. 'FEA-(\\d+\\d\\d)'",
    "property 'n': a naming pattern gives text: the property's schema is Schemas.OfStr"]));
}

## IDN-03 · Every store holds the path schemas and `Schemas.HasInstance`; the members of its own relations are owned

In [ ]:
{
  for (const name of ["Paths.OfRoot", "Paths.OfEntry", "Paths.OfTarget", "Paths.Next"]) {
    assert(store.registered(name).validate().length === 0 && store.framework(store.registered(name)));
  }
  assert(same([...store.Paths.OfRoot.as_schema().properties.keys()], ["singleton", "next"]));
  assert(same([...store.Paths.OfEntry.as_schema().properties.keys()], ["i", "o", "properties", "next"]));
  assert(same(store.Paths.Next.as_schema().branches.map((b: any) => b.name), ["entry", "target"]));
  const other: any = new B.Store();
  assert(other.Paths.OfRoot.as_schema() !== store.Paths.OfRoot.as_schema()); // each store's own
  const HasInstance = store.Schemas.HasInstance.as_schema();
  assert(same([...HasInstance.entry.links.keys()], ["schema", "instance"]) && same([...HasInstance.entry.properties.keys()], ["name"]));
  assert(store.Schemas.OfObject.as_schema().adjacencies.get("instances").relation === HasInstance);
  const owning = store.Schemas.OfProperty.as_schema().anonymous.filter((a: any) => a.owner).map((a: any) => `${a.relation.name} ${a.me}`);
  assert(same(owning, ["Schemas.HasProperty property"]));
  assert(store.Schemas.OfObject.as_schema().anonymous.filter((a: any) => a.relation.name === "Schemas.Defines").every((a: any) => a.owner));
}

## IDN-04 · An object's path on demand: read off its ownership entries back to a singleton, a nested value; refused where it has no owner, or more than one

In [ ]:
const features = store.Instances.wts.Features;
const toPlain = (value: unknown) => Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), value);
{
  assert(features === store.singleton("wts.Features"));
  store.wts.Directory(features, (f: any) => f.dir((a: any) => a.insert((e: any) => e.name("DEF-693").defn((d: any) => (
    d
      .title("Has 88 keys")
      .clauses((c: any) => c.insert((h: any) => h.name("C-1").clause((x: any) => x)))
  )))));
  const definition = features.dir({ name: "DEF-693" });
  const clause = definition.clauses({ name: "C-1" });
  assert(equal(toPlain(Paths.of(definition)), map({ singleton: "wts.Features", next: { entry: {
    i: "dir", o: "def", properties: { name: "DEF-693" }, next: { target: {} } } } })));
  assert(equal(toPlain(clause.as_identity()), map({ singleton: "wts.Features", next: { entry: {
    i: "dir", o: "def", properties: { name: "DEF-693" }, next: { entry: {
      i: "clauses", o: "", properties: { name: "C-1" }, next: { target: {} } } } } } }))); // an anonymous owner
  assert(equal(toPlain(Paths.of(features)), map({ singleton: "wts.Features", next: { target: {} } })));
  assert(Paths.same(Paths.of(clause), clause.as_identity()) && !Paths.same(Paths.of(clause), Paths.of(definition)));
  assert(Paths.same(null, null) && !Paths.same(Paths.of(clause), null));
  store.wts.Definition(definition, (d: any) => d.clauses((a: any) => a.remove((e: any) => e.clause(clause))));
  raises(LookupError, () => Paths.of(clause), "a 'wts.Clause' has no owner: its path is not known"); // its owning entry removed
  const Twice = store.Schemas.OfObject((o: any) => o.name("wts.Twice").ref(true));
  const Holds = store.Schemas.OfRelation((r: any) => r.name("wts.Holds").links([(t: any) => t.name("a"), (t: any) => t.name("b")])
    .uniques([["a"], ["b"]]));
  store.Schemas.OfObject(Twice, (o: any) => o.adjacencies([
    (t: any) => t.name("up").relation([Holds]).me("b").owner(true),
    (t: any) => t.name("down").relation([Holds]).me("a"),
  ]));
  const [first] = store._writing(() => { // within one write, objects are given their owners when it ends: here, each other
    const [first, second] = [store.wts.Twice((t: any) => t), store.wts.Twice((t: any) => t)];
    store.wts.Twice(first, (t: any) => t.up((a: any) => a.insert((e: any) => e.a(second))));
    store.wts.Twice(second, (t: any) => t.up((a: any) => a.insert((e: any) => e.a(first))));
    return [first, second];
  });
  raises(LookupError, () => Paths.of(first), "an ownership cycle: no singleton owns the object");
  const pair = [0, 1].map(() => store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("a"), (t: any) => t.name("b")])
    .uniques([["a"], ["b"]])));
  store.Schemas.OfObject(Directory, (o: any) => o.adjacencies([(t: any) => t.relation([pair[0]]).me("a"),
    (t: any) => t.name("rights").relation([pair[1]]).me("a")]));
  store.Schemas.OfObject((o: any) => o.name("wts.Split").ref(true).adjacencies([
    (t: any) => t.name("left").relation([pair[0]]).me("b").owner(true),
    (t: any) => t.name("right").relation([pair[1]]).me("b").owner(true),
  ]));
  const split = store.wts.Split((s: any) => s.left((a: any) => a.insert((e: any) => e.a(features))));
  assert(equal(toPlain(Paths.of(split)), map({ singleton: "wts.Features", next: { entry: {
    i: "", o: "left", properties: {}, next: { target: {} } } } }))); // the owner's adjacency anonymous
  store.wts.Split(split, (s: any) => s.right((a: any) => a.insert((e: any) => e.a(features))));
  raises(LookupError, () => Paths.of(split), "a 'wts.Split' has more than one owner: its path is not known");
  raises(ReadOnlyError, () => store.wts.Twice(first, (t: any) => t.as_identity("twice-9")),
    "an identity is read-only: it is the object's path, read off its ownership");
}

## IDN-05 · An object built without an owner is given the one its schema allows: a singleton's, its keys from their naming patterns; else refused

In [ ]:
{
  store.wts.Directory(features, (f: any) => f.dir((a: any) => a.insert((e: any) => e.defn((d: any) => d.title("nameless"))))); // no key
  const made = store.wts.Definition((d: any) => d.title("made alone")); // its owner the one singleton that can be, Features
  assert(features.dir({ name: "DEF-694" }) === made); // its key the next its pattern gives, past every number held
  assert(store.wts.Definition((d: any) => d) === features.dir({ name: "DEF-695" }));
  raises(ValueError, () => store.wts.Clause((c: any) => c), // a definition owns it, and none is a singleton
    "a 'wts.Clause' needs an owner through 'wts.HasClause': give it one, since no singleton can be");
  raises(ValueError, () => store.wts.Split((s: any) => s),
    "a 'wts.Split' needs an owner through 'left', 'right': give it one, since more than one singleton can be");
  const Notes = store.Schemas.OfRelation((r: any) => (
    r
      .name("wts.Notes")
      .links([(t: any) => t.name("directory"), (t: any) => t.name("note")])
      .properties([(t: any) => t.name("name").schema([Text])]) // a key with no naming pattern
      .uniques([["directory", "name"], ["note"]])
  ));
  store.Schemas.OfObject(Directory, (o: any) => o.adjacencies([(t: any) => t.name("notes").relation([Notes]).me("directory")]));
  store.Schemas.OfObject((o: any) => o.name("wts.Note").ref(true).adjacencies([
    (t: any) => t.relation([Notes]).me("note").owner(true),
  ]));
  raises(ValueError, () => store.wts.Note((n: any) => n),
    "a 'wts.Note' needs an owner: the key 'name' of 'wts.Notes' has no naming pattern to give it a value");
  store.wts.Directory(features, (f: any) => f.notes((a: any) => a.insert((e: any) => e.name("first").note((n: any) => n))));
  assert(features.notes({ name: "first" }).as_schema() === store.registered("wts.Note"));
}

## IDN-06 · Lookup by keys: on any adjacency, and on `store.Instances` for an object its schema owns; refused where nothing has them

In [ ]:
{
  store.Schemas.OfObject((o: any) => o.name("wts.Free").ref(true).pattern(String.raw`free-(\d+)`));
  const free = store.wts.Free((f: any) => f); // no owner() adjacency: its schema owns it, by its key
  assert(store.Instances.wts.Free({ name: "free-1" }) === free);
  assert(equal(toPlain(Paths.of(free)), map({ singleton: ".", next: { entry: {
    i: "schemas", o: "", properties: { name: "wts" }, next: { entry: {
      i: "schemas", o: "", properties: { name: "Free" }, next: { entry: {
        i: "instances", o: "", properties: { name: "free-1" }, next: { target: {} } } } } } } } })));
  assert(repr(store.Instances.wts) === "<Instances.wts>");
  assert(store.Instances.wts.then === undefined && store.Instances.wts[Symbol.iterator] === undefined); // probes name no part
  const Loose = new B.ObjectBuilder().name("wts.Loose").ref(true).create(); // made outside any store, as an adapter makes it
  store._register(Loose); // registered, though no module defines it: its path is the one its name spells
  assert(equal(toPlain(Paths.of(store.wts.Loose((l: any) => l))), owned("wts.Loose", "loose-1")));
  raises(LookupError, () => store.Instances.wts.Free({ name: "free-9" }), "no 'wts.Free' has the key 'free-9'");
  raises(LookupError, () => store.Instances.wts.Definition({ name: "definition-1" }), // a definition is owned through Defines, not its schema
    "no 'wts.Definition' has the key 'definition-1'");
  raises(KeyError, () => store.Instances.wts.Free("free-1"), "an object its schema owns is found by its key, {'name': ...}; got 'free-1'");
  raises(AttributeError, () => store.Instances._hidden, "_hidden");
  raises(LookupError, () => features.dir({ name: "DEF-999" }), "no entry of 'dir' has the keys {'name': 'DEF-999'}");
  raises(KeyError, () => features.dir({ nope: 1n }), "'nope' is not a property of 'wts.Defines'");
  const Trio = store.Schemas.OfRelation((r: any) => r.name("wts.Trio").links(["a", "b", "c"].map((n) => (t: any) => t.name(n))));
  store.Schemas.OfObject(Directory, (o: any) => o.adjacencies([(t: any) => t.name("trio").relation([Trio]).me("a")]));
  raises(TypeError, () => features.trio({}), "a lookup through 'trio' needs a relation of two links");
}

## IDN-07 · Owned objects in snapshots and logs: listed at their paths, read back owners first, each owned by the entry its path's last step gives

In [ ]:
{
  /** wts.Features, defining DEF-001, which holds the clause C-1. */
  const defined = (store: any): any => {
    const features = store.Instances.wts.Features;
    store.wts.Directory(features, (f: any) => f.dir((a: any) => a.insert((e: any) => e.name("DEF-001").defn((d: any) => (
      d
        .title("Keys")
        .clauses((c: any) => c.insert((h: any) => h.name("C-1").clause((x: any) => x)))
    )))));
    return features;
  };
  const [source, target]: any[] = [new B.Store(), new B.Store()];
  wts(source);
  wts(target);
  const features = defined(source);
  const graph: any = Plain.ToPlain(source).Reachable(source.registered("wts.Directory"), features);
  assert(same(graph.get("objects").map((o: any) => o.get("schema")), ["wts.Directory", "wts.Definition", "wts.Clause"]));
  assert(equal(graph.get("objects")[1].get("path"), map({ singleton: "wts.Features", next: { entry: {
    i: "dir", o: "def", properties: { name: "DEF-001" }, next: { target: {} } } } })));
  const back: any = Plain.FromPlain(target).Reachable(target.registered("wts.Directory"), graph); // owners first; each owned by its path
  assert(back === target.Instances.wts.Features && `${back.dir({ name: "DEF-001" }).title}` === "Keys");
  assert(equal(Plain.ToPlain(target).Reachable(target.registered("wts.Directory"), back), graph));
  assert(Plain.FromPlain(target).Reachable(target.registered("wts.Directory"), graph) === back && back.dir.length === 1); // an update
  const clause = features.dir({ name: "DEF-001" }).clauses({ name: "C-1" });
  raises(DecodeError, () => Plain.FromPlain(target).OfObject(target.registered("wts.Clause"),
    Plain.ToPlain(source).OfObject(source.registered("wts.Clause"), clause)), "$.objects: an object's owner is not in the snapshot");
  const renamed = Json.loads(Json.dumps(graph).replaceAll('"o": "def"', '"o": "nope"'));
  raises(DecodeError, () => Plain.FromPlain(target).Reachable(target.registered("wts.Directory"), renamed),
    "'wts.Definition' has no adjacency 'nope' marked owner() for a path to go through");

  // A log writes them with their paths, and reads them back where their paths place them.
  const log = Plain.ToPlain(source).Snapshot();
  const third: any = new B.Store();
  Plain.FromPlain(third).Snapshot(log);
  assert(Json.dumps(Plain.ToPlain(third).Snapshot() as never) === Json.dumps(log as never)
    && Plain.FromPlain(third).Snapshot(log).length === 0); // again: each exists
  assert(third.Instances.wts.Features.dir({ name: "DEF-001" }).clauses({ name: "C-1" }).as_schema() === third.registered("wts.Clause"));
  const at = graph.get("objects")[1].get("path");
  for (const [definition, says] of [
    [map({ key: "wts.Clause/x" }), "'wts.Clause/x': data is created with its path and its schema, not a key"],
    [new Map([["path", at]]), "a data object's definition names its schema"],
    [new Map([["path", at], ["schema", "Paths.OfTarget"]]), "'Paths.OfTarget' is not a reference object schema"],
    [new Map([["path", Json.loads(Json.dumps(at).replace("DEF-001", "DEF-999"))], ["schema", "wts.Definition"]]), null],
    [map({ path: { singleton: "wts.Features", next: { entry: { i: "dir", o: "def", properties: { name: "DEF-998" },
      next: { entry: { i: "clauses", o: "", properties: { name: "C-1" }, next: { target: {} } } } } } }, schema: "wts.Clause" }),
      "the object's owner is not in the store"],
  ] as [unknown, string | null][]) {
    const one = [new Map([["exists", new Map([["definition", definition]])]])];
    if (says === null) {
      Plain.FromPlain(third).Snapshot(one); // a new definition, owned as its path says
      assert(third.Instances.wts.Features.dir({ name: "DEF-999" }).as_schema() === third.registered("wts.Definition"));
      continue;
    }
    raises(DecodeError, () => Plain.FromPlain(third).Snapshot(one), says);
  }
  const another = Json.loads(Json.dumps(graph).replaceAll("DEF-001", "DEF-002")); // the store's features hold DEF-001 and DEF-999
  const Directory3 = third.registered("wts.Directory");
  assert(Plain.FromPlain(third).Reachable(Directory3, another) === third.Instances.wts.Features);
  assert(same(third.Instances.wts.Features.dir.map((e: any) => `${e.name}`), ["DEF-001", "DEF-999", "DEF-002"])); // a new one, owned
  const atFeatures = { singleton: "wts.Features", next: { target: {} } };
  raises(DecodeError, () => Plain.FromPlain(third).OfObject(third.registered("wts.Definition"),
    map({ root: atFeatures, objects: [{ path: atFeatures, schema: "wts.Definition", properties: {} }] })),
  "$.objects[0].path: a 'wts.Definition' is not a singleton: its path goes through what owns it");
}